# 38 - Learned Fusion Ranker retrained against the new, non-circular gold standard

The original Learned Fusion Ranker (notebook 30) was trained and evaluated against labels derived from `goi_search_results.json`, production's own circular output. This retrains the exact same architecture (same 10 features, same 5-fold GroupKFold cross-validation grouped by query_id) but against `final_gold_labels.json`, the pooled, multi-judge, human-tie-broken gold standard from notebooks 34-35 instead.

Only covers the 5 pilot queries (417 of 419 gold-labeled candidates have existing features from the original fusion ranker's candidate pool, 2 are dropped for missing features). This is a proof of concept at small scale, not the full 101-query result yet.

Key question: does the feature-importance finding from notebook 30 (MiniLM's inverse rank mattering more than the pretrained reranker's own score) still hold against a gold standard that isn't circular?

In [5]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

OUTPUT_DIR = Path("result/38_fusion_ranker_new_gold")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RELEVANT_THRESHOLD = 2  # gold_label >= this counts as "relevant" for the binary classifier target
# NOTE: was 1 -- changed to 2 (strict, "highly relevant" only) after the first run showed 409/417 (98%)
# labeled relevant at threshold 1, only 8 negatives total across 5 queries. That's too imbalanced for
# 5-fold GroupKFold to estimate stable coefficients from -- threshold 2 gives 292 positive vs 125
# negative, a real, learnable split, instead of a near-degenerate one.

gold = pd.DataFrame(json.load(open("result/35_llm_judge_ensemble/final_gold_labels.json")))
features = pd.read_csv("result/30_learned_fusion_ranker/scored_candidates.csv")

feat_df = gold.merge(features, on=["query_id", "domain"], how="inner")
print(f"Gold-labeled candidates: {len(gold)}")
print(f"Matched to existing features: {len(feat_df)} ({len(gold) - len(feat_df)} dropped, missing features)")

feat_df["relevant"] = (feat_df["gold_label"] >= RELEVANT_THRESHOLD).astype(int)
print(f"Relevant (gold_label >= {RELEVANT_THRESHOLD}): {feat_df['relevant'].sum()}/{len(feat_df)}")

Gold-labeled candidates: 419
Matched to existing features: 417 (2 dropped, missing features)
Relevant (gold_label >= 2): 291/417


In [6]:
feature_cols = ['score_minilm', 'score_linq', 'score_gte', 'score_bm25',
                'invrank_minilm', 'invrank_linq', 'invrank_gte', 'invrank_bm25',
                'reranker_score', 'n_channels']
X = feat_df[feature_cols].values
y = feat_df["relevant"].values
groups = feat_df["query_id"].values
n_queries = feat_df["query_id"].nunique()

gkf = GroupKFold(n_splits=n_queries)  # exactly 5 pilot queries -- this is leave-one-query-out
feat_df["score_logreg_new"] = np.nan
feat_df["score_gbdt_new"] = np.nan

logreg_coefs = []
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    Xtr, Xte = X[train_idx], X[test_idx]
    ytr = y[train_idx]

    lr = LogisticRegression(max_iter=2000, class_weight="balanced")
    lr.fit(Xtr, ytr)
    feat_df.loc[feat_df.index[test_idx], "score_logreg_new"] = lr.predict_proba(Xte)[:, 1]
    logreg_coefs.append(lr.coef_[0])

    gbdt = HistGradientBoostingClassifier(max_iter=150, max_depth=4, class_weight="balanced", random_state=fold)
    gbdt.fit(Xtr, ytr)
    feat_df.loc[feat_df.index[test_idx], "score_gbdt_new"] = gbdt.predict_proba(Xte)[:, 1]
    print(f"[Train] fold {fold+1}/{n_queries} done -- {len(test_idx)} held-out rows scored (query held out: {groups[test_idx][0]})")

print()
print("Cross-validated feature weights (LogisticRegression, averaged across folds), new gold standard:")
avg_coefs = np.mean(logreg_coefs, axis=0)
for name, coef in sorted(zip(feature_cols, avg_coefs), key=lambda x: -abs(x[1])):
    print(f"    {name:<16} {coef:+.3f}")

[Train] fold 1/5 done -- 91 held-out rows scored (query held out: 4)
[Train] fold 2/5 done -- 89 held-out rows scored (query held out: 1)
[Train] fold 3/5 done -- 85 held-out rows scored (query held out: 2)
[Train] fold 4/5 done -- 82 held-out rows scored (query held out: 5)
[Train] fold 5/5 done -- 70 held-out rows scored (query held out: 3)

Cross-validated feature weights (LogisticRegression, averaged across folds), new gold standard:
    score_linq       +1.965
    invrank_bm25     -1.252
    invrank_minilm   -0.909
    score_gte        +0.791
    invrank_linq     +0.416
    invrank_gte      +0.323
    reranker_score   +0.319
    n_channels       -0.289
    score_minilm     -0.146
    score_bm25       -0.134


In [7]:
# Compare to the ORIGINAL feature weights (trained against the old, circular ground truth).
old_weights = json.load(open("result/30_learned_fusion_ranker/feature_weights.json"))
print("Original weights (old, circular ground truth):")
for name, coef in sorted(old_weights.items(), key=lambda x: -abs(x[1])):
    print(f"    {name:<16} {coef:+.3f}")
print()
print("If the ranking of feature importance above looks similar to the new weights printed in the previous cell,")
print("that's a good sign the earlier interpretive finding wasn't just an artifact of the circular ground truth.")

Original weights (old, circular ground truth):
    invrank_minilm   +5.014
    score_linq       +1.097
    invrank_gte      +0.958
    score_gte        +0.921
    invrank_linq     +0.867
    score_minilm     +0.749
    n_channels       +0.459
    invrank_bm25     -0.258
    reranker_score   +0.164
    score_bm25       +0.054

If the ranking of feature importance above looks similar to the new weights printed in the previous cell,
that's a good sign the earlier interpretive finding wasn't just an artifact of the circular ground truth.


In [8]:
def ndcg_at_k(retrieved_domains, graded_scores, k):
    dcg = sum(graded_scores.get(d, 0) / np.log2(i + 2) for i, d in enumerate(retrieved_domains[:k]))
    ideal = sorted(graded_scores.values(), reverse=True)[:k]
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg else 0.0


K_VALUES = [5, 10, 20]
rows = []
for query_id in sorted(feat_df["query_id"].unique()):
    sub = feat_df[feat_df["query_id"] == query_id]
    graded = dict(zip(sub["domain"], sub["gold_label"]))

    ranked_new = sub.sort_values("score_gbdt_new", ascending=False)["domain"].tolist()
    ranked_old = sub.sort_values("score_gbdt", ascending=False)["domain"].tolist()  # original model's held-out score, same rows

    for k in K_VALUES:
        rows.append({
            "query_id": query_id, "k": k,
            "ndcg_retrained_on_new_gold": ndcg_at_k(ranked_new, graded, k),
            "ndcg_original_model_scores": ndcg_at_k(ranked_old, graded, k),
        })

ndcg_df = pd.DataFrame(rows)
ndcg_df.to_csv(OUTPUT_DIR / "ndcg_comparison.csv", index=False)
print("NDCG against the new gold standard, per k, averaged across the 5 pilot queries:")
print(ndcg_df.groupby("k")[["ndcg_retrained_on_new_gold", "ndcg_original_model_scores"]].mean().round(3))
print()
print("'ndcg_original_model_scores' = the OLD model's held-out scores (trained on circular ground truth),")
print("ranked and scored against the NEW gold labels -- shows how much retraining on the better labels actually helped.")

NDCG against the new gold standard, per k, averaged across the 5 pilot queries:
    ndcg_retrained_on_new_gold  ndcg_original_model_scores
k                                                         
5                        0.934                       0.915
10                       0.952                       0.930
20                       0.947                       0.940

'ndcg_original_model_scores' = the OLD model's held-out scores (trained on circular ground truth),
ranked and scored against the NEW gold labels -- shows how much retraining on the better labels actually helped.
